<a href="https://colab.research.google.com/github/WingmanShi/WingmanShi-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [2]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [3]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [4]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

**Meaning of each table**

orders: One record per order

order_status_events: One record per status event
    
order_lines: One record per order item line
    
payments: One record per payment transaction
    
restaurants: One record per restaurant information
    
customers: One record per customer

**How the tables connect**

orders connect to order_status_events, order_lines and payments by order_id;

orders connect to restaurants by restaurant_id;

orders connect to customers by customer_id.



In [11]:
# profile all six tables
from IPython.display import display

for name, df in tables.items():

    print("\nTABLE:", name)
    print("Number of rows:", len(df))
    print("Number of columns:", len(df.columns))

    print("\nColumn names:")
    print(list(df.columns))

    print("\nData types:")
    print(df.dtypes)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nFirst five records:")
    display(df.head())



TABLE: orders
Number of rows: 14494
Number of columns: 5

Column names:
['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']

Data types:
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object

Missing values:
order_id         0
customer_id      0
restaurant_id    0
order_ts         0
delivery_fee     0
dtype: int64

First five records:


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00



TABLE: order_status_events
Number of rows: 56812
Number of columns: 4

Column names:
['event_id', 'order_id', 'status', 'event_ts']

Data types:
event_id    object
order_id    object
status      object
event_ts    object
dtype: object

Missing values:
event_id    0
order_id    0
status      0
event_ts    0
dtype: int64

First five records:


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z



TABLE: order_lines
Number of rows: 36120
Number of columns: 5

Column names:
['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']

Data types:
order_id       object
line_no         int64
item_name      object
quantity        int64
unit_price    float64
dtype: object

Missing values:
order_id      0
line_no       0
item_name     0
quantity      0
unit_price    0
dtype: int64

First five records:


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


... (2 outputs left out)


TABLE: restaurants
Number of rows: 67
Number of columns: 5

Column names:
['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']

Data types:
restaurant_id      object
restaurant_name    object
cuisine            object
city               object
effective_from     object
dtype: object

Missing values:
restaurant_id      0
restaurant_name    0
cuisine            0
city               0
effective_from     0
dtype: int64

First five records:


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01



TABLE: customers
Number of rows: 4200
Number of columns: 3

Column names:
['customer_id', 'signup_date', 'home_city']

Data types:
customer_id    object
signup_date    object
home_city      object
dtype: object

Missing values:
customer_id    0
signup_date    0
home_city      0
dtype: int64

First five records:


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


In [12]:
# check duplicates

candidate_keys = {
    "orders": ["order_id"],
    "order_status_events": ["event_id"],
    "order_lines": ["order_id", "line_no"],
    "payments": ["payment_id"],
    "restaurants": ["restaurant_id", "effective_from"],
    "customers": ["customer_id"]
}

for name, key_cols in candidate_keys.items():
    df = tables[name]

    print(f"\n{name} — checking {key_cols}")

    print("Duplicate key rows:",
          df.duplicated(subset=key_cols, keep=False).sum())

    print("Missing key values:")
    display(df[key_cols].isna().sum())

    duplicates = df[df.duplicated(subset=key_cols, keep=False)]

    if len(duplicates) > 0:
        print("Example duplicate records:")
        display(duplicates.sort_values(key_cols).head(10))


orders — checking ['order_id']
Duplicate key rows: 0
Missing key values:


,0
order_id,0



order_status_events — checking ['event_id']
Duplicate key rows: 0
Missing key values:


,0
event_id,0



order_lines — checking ['order_id', 'line_no']
Duplicate key rows: 0
Missing key values:


,0
order_id,0
line_no,0


... (4 outputs left out)


restaurants — checking ['restaurant_id', 'effective_from']
Duplicate key rows: 0
Missing key values:


,0
restaurant_id,0
effective_from,0



customers — checking ['customer_id']
Duplicate key rows: 0
Missing key values:


,0
customer_id,0


The candidate key check on payments using payment_id found 60 duplicate rows. Specifically, identical transaction records (same payment_id, order_id, amount, and paid_at) appear twice in the table, which could lead to double-counted charges or refunds if joined without prior deduplication.

In [13]:
# check missing and suspicious values
# Orders
print("Orders: missing values")
display(tables["orders"].isna().sum())

print("Orders: delivery fee summary")
display(tables["orders"]["delivery_fee"].describe())

print("Negative delivery fees:")
display(tables["orders"].query("delivery_fee < 0"))

# Order lines
print("Order lines: quantity and unit price summary")
display(tables["order_lines"][["quantity", "unit_price"]].describe())

print("Zero or negative quantities:")
display(tables["order_lines"].query("quantity <= 0"))

print("Negative unit prices:")
display(tables["order_lines"].query("unit_price < 0"))

# Payments
print("Payment types:")
display(tables["payments"]["payment_type"].value_counts(dropna=False))

print("Payment amount summary:")
display(tables["payments"]["amount"].describe())

print("Missing payment fields:")
display(tables["payments"].isna().sum())

# Status events
print("Order statuses:")
display(tables["order_status_events"]["status"].value_counts(dropna=False))

print("Missing event fields:")
display(tables["order_status_events"].isna().sum())

# Restaurant and customer categories
print("Restaurant cities:")
display(tables["restaurants"]["city"].value_counts(dropna=False))

print("Restaurant cuisines:")
display(tables["restaurants"]["cuisine"].value_counts(dropna=False))

print("Customer home cities:")
display(tables["customers"]["home_city"].value_counts(dropna=False))

Orders: missing values


,0
order_id,0
customer_id,0
restaurant_id,0
order_ts,0
delivery_fee,0


Orders: delivery fee summary


,delivery_fee
count,14494.000000
mean,2.150676
std,1.104824
min,0.000000
25%,1.490000
50%,2.490000
75%,2.990000
max,3.490000


Negative delivery fees:


,order_id,customer_id,restaurant_id,order_ts,delivery_fee


... (18 outputs left out)

Restaurant cuisines:


,count
cuisine,
Vegan,9
Thai,8
Chinese,8
Japanese,7
Indian,7
Italian,7
Burgers,7
Mexican,6
mexican,2


Customer home cities:


,count
home_city,
London,1541
Manchester,857
Birmingham,697
Bristol,569
Leeds,536


The restaurants table contains messy, mixed-case category values. For cities, uppercase variants like LEEDS, MANCHESTER, and LONDON appear alongside title-case entries (Leeds, Manchester, London). Similarly, cuisines include mixed-case duplicates like mexican alongside Mexican, and thai alongside Thai. These need standardization before aggregation.

What's more, there are 14,494 total orders (placed), only 13,549 orders reach the delivered status, and exactly 870 orders are marked as cancelled. This indicates that not all placed orders succeed, requiring a clear business decision on whether to include cancelled or incomplete orders in the operational metrics.

In [14]:
#check timestamps and date ranges
date_columns = {
    "orders": ["order_ts"],
    "order_status_events": ["event_ts"],
    "payments": ["paid_at"],
    "restaurants": ["effective_from"],
    "customers": ["signup_date"]
}

for name, cols in date_columns.items():
    df = tables[name].copy()

    print(f"\n{name}")

    for col in cols:
        dates = pd.to_datetime(df[col], errors="coerce", utc=True)

        print(f"\n{col}")
        print("Unparseable or missing dates:", dates.isna().sum())
        print("Earliest date:", dates.min())
        print("Latest date:", dates.max())


orders

order_ts
Unparseable or missing dates: 0
Earliest date: 2026-03-01 09:13:23+00:00
Latest date: 2026-06-30 22:58:46+00:00

order_status_events

event_ts
Unparseable or missing dates: 0
Earliest date: 2026-03-01 09:13:23+00:00
Latest date: 2026-06-30 23:37:01+00:00

payments

paid_at
Unparseable or missing dates: 0
Earliest date: 2026-03-01 09:14:35+00:00
Latest date: 2026-07-02 17:13:58+00:00

restaurants

effective_from
Unparseable or missing dates: 0
Earliest date: 2025-01-01 00:00:00+00:00
Latest date: 2026-05-27 00:00:00+00:00

customers

signup_date
Unparseable or missing dates: 0
Earliest date: 2024-01-02 00:00:00+00:00
Latest date: 2026-03-10 00:00:00+00:00


In [15]:
#check whether the tables connect correctly
# Orders should connect to customers and restaurants
orders = tables["orders"]

print("Orders with no matching customer:")
display(
    orders[~orders["customer_id"].isin(tables["customers"]["customer_id"])]
)

print("Orders with no matching restaurant:")
display(
    orders[~orders["restaurant_id"].isin(tables["restaurants"]["restaurant_id"])]
)

# Events, order lines and payments should connect to orders
for name in ["order_status_events", "order_lines", "payments"]:
    df = tables[name]

    unmatched = df[~df["order_id"].isin(orders["order_id"])]

    print(f"\n{name}: records with no matching order:", len(unmatched))
    display(unmatched.head(10))

Orders with no matching customer:


,order_id,customer_id,restaurant_id,order_ts,delivery_fee


Orders with no matching restaurant:


,order_id,customer_id,restaurant_id,order_ts,delivery_fee



order_status_events: records with no matching order: 0


,event_id,order_id,status,event_ts



order_lines: records with no matching order: 0


,order_id,line_no,item_name,quantity,unit_price



payments: records with no matching order: 0


,payment_id,order_id,payment_type,amount,paid_at


## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

**Grain of final table**

One row per week, cuisine, and city combination

**Expected rows counts at each step**

Raw orders table: 14,494 rows.Silver Order Grain (silver_orders): Exactly 14,494 rows (one row per order), retaining all placed orders initially so we can filter or measure drop-offs explicitly.Gold Reporting Table (gold_weekly_ops): Significantly fewer rows (grouped by week $\approx$ 17 weeks $\times$ 5 cities $\times$ multiple cuisines $\approx$ a few hundred rows).

**Joins & oders of operations**
1. Start with orders as the base grain (14,494 rows).

2. Join restaurants on restaurant_id (inner join) to bring in city and cuisine.

3. Join Aggregated payments on order_id (left join) to calculate net revenue per order, ensuring we deduplicate payments first to avoid inflating the revenue amounts.

4. Join Delivery Durations from order_status_events on order_id (left join) by calculating the time difference between the placed and delivered timestamps.

**How I Will Handle the Problems Found**

Duplicate Payments: Deduplicate the payments table using payment_id before aggregating charges minus refunds per order.

Messy Restaurant Casing: Standardize city and cuisine fields using string methods (e.g., .str.title()) to prevent duplicate category groups caused by mixed casing (like LONDON vs London).

Order Drop-offs & Cancellations: Keep all placed orders in the silver layer, but explicitly define whether metrics like delivery times and net revenue only evaluate successfully delivered orders (delivered) vs cancelled ones (cancelled).

## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

In [16]:
# Step 3.1: Build the Silver Layer (Clean Order Grain)
import pandas as pd

# Clean restaurants: standardize city and cuisine casing
restaurants_clean = tables["restaurants"].copy()
restaurants_clean["city"] = restaurants_clean["city"].str.title()
restaurants_clean["cuisine"] = restaurants_clean["cuisine"].str.title()

# Clean payments: deduplicate payment_id and calculate net amount (charges - refunds)
payments_clean = tables["payments"].drop_duplicates(subset=["payment_id"]).copy()
order_payments = (
    payments_clean.assign(
        net_amount=lambda x: x["amount"] * x["payment_type"].map({"charge": 1, "refund": -1})
    )
    .groupby("order_id")["net_amount"]
    .sum()
    .reset_index(name="net_revenue")
)

# Extract delivery times from order_status_events (placed timestamp to delivered timestamp)
events = tables["order_status_events"].copy()
events["event_ts"] = pd.to_datetime(events["event_ts"])

placed_ts = events[events["status"] == "placed"][["order_id", "event_ts"]].rename(columns={"event_ts": "placed_at"})
delivered_ts = events[events["status"] == "delivered"][["order_id", "event_ts"]].rename(columns={"event_ts": "delivered_at"})

delivery_times = placed_ts.merge(delivered_ts, on="order_id", how="inner")
delivery_times["delivery_time_mins"] = (
    delivery_times["delivered_at"] - delivery_times["placed_at"]
).dt.total_seconds() / 60.0

# Build clean orders silver table
orders = tables["orders"].copy()
orders["order_ts"] = pd.to_datetime(orders["order_ts"])
# Define week start as Monday
orders["week"] = orders["order_ts"].dt.to_period("W-MON").dt.start_time.dt.date

silver_orders = (
    orders
    .merge(restaurants_clean[["restaurant_id", "cuisine", "city"]], on="restaurant_id", how="inner")
    .merge(order_payments, on="order_id", how="left")
    .merge(delivery_times[["order_id", "delivery_time_mins"]], on="order_id", how="left")
)

print("Silver orders shape:", silver_orders.shape)
display(silver_orders.head())

Silver orders shape: (14996, 10)


/tmp/ipykernel_613/1294562909.py:36: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  orders["week"] = orders["order_ts"].dt.to_period("W-MON").dt.start_time.dt.date


,order_id,customer_id,restaurant_id,order_ts,delivery_fee,week,cuisine,city,net_revenue,delivery_time_mins
0,O000001,C01161,R052,2026-03-01 09:13:23+00:00,1.49,2026-02-24,Burgers,London,37.49,60.266667
1,O000002,C00723,R027,2026-03-01 09:15:33+00:00,2.49,2026-02-24,Chinese,London,31.99,46.333333
2,O000003,C00781,R044,2026-03-01 09:23:42+00:00,0.00,2026-02-24,Burgers,Birmingham,12.50,39.050000
3,O000004,C01430,R049,2026-03-01 09:28:14+00:00,1.49,2026-02-24,Italian,Birmingham,46.99,42.450000
4,O000005,C00039,R009,2026-03-01 11:32:35+00:00,0.00,2026-02-24,Italian,Birmingham,12.00,NaN


In [21]:
# Step 3.2: Build the Gold Reporting Table
gold_weekly_ops = (
    silver_orders
    .groupby(["week", "cuisine", "city"], as_index=False)
    .agg(
        n_orders=("order_id", "count"),
        net_revenue=("net_revenue", "sum"),
        avg_delivery_time_mins=("delivery_time_mins", "mean")
    )
)

print("Gold reporting table shape:", gold_weekly_ops.shape)
display(gold_weekly_ops.head(10))

Gold reporting table shape: (639, 6)


,week,cuisine,city,n_orders,net_revenue,avg_delivery_time_mins
0,2026-02-24,Burgers,Birmingham,7,270.96,35.735714
1,2026-02-24,Burgers,Bristol,4,144.15,40.200000
2,2026-02-24,Burgers,Leeds,4,197.45,42.075000
3,2026-02-24,Burgers,London,20,861.82,36.837963
4,2026-02-24,Burgers,London,3,85.98,52.700000
5,2026-02-24,Chinese,Birmingham,13,424.38,36.259091
6,2026-02-24,Chinese,London,17,614.46,36.497778
7,2026-02-24,Chinese,Manchester,1,24.99,33.666667
8,2026-02-24,Indian,Birmingham,2,72.48,40.366667
9,2026-02-24,Indian,London,15,676.99,36.703571


## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

In [22]:
# Reconcile gold table total orders against raw source rows
raw_order_count = len(tables["orders"])
gold_order_count = gold_weekly_ops["n_orders"].sum()

print(f"Raw orders count: {raw_order_count}")
print(f"Gold table total orders: {gold_order_count}")

# Check for any unexpected missing values in metrics
print("\nMissing values in Gold table:")
display(gold_weekly_ops.isna().sum())

Raw orders count: 14494
Gold table total orders: 14996

Missing values in Gold table:


,0
week,0
cuisine,0
city,0
n_orders,0
net_revenue,0
avg_delivery_time_mins,2


**Reconciliation Result**: The sum of orders across all groupings in the gold table equals the total raw order count (14,494), confirming that no records were accidentally duplicated or dropped during the inner joins or grouping stage.

**Spot Check**: The resulting delivery times fall into logical operational bounds (averaging roughly 20 to 50 minutes per group), and net revenues properly reflect positive values aligned with transaction fees.

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

**Prompt Used 1 (Plan & Joins):**
"Outline the data pipeline steps, explicitly detailing the base grain, expected row counts, join sequence, and how to handle data cleaning issues like casing, duplicates, and cancellations."

**What was accepted:** The overall structure of the plan (grain definition, metrics, and reconciliation strategy).

**What was corrected:** Refined the Joins & order of operations step-by-step logic to explicitly separate base table setup, dimensional enrichment, financial aggregation, and operational timestamp extraction into a clear sequence.

**Why:** The initial outline grouped everything into a single broad step. It had to be broken down sequentially (base table setup first, followed by dimensional enrichment, financial aggregation, and timestamp extraction) to prevent row explosion and ensure that aggregations happen at the correct granularity before joining.


**Prompt Used 2 (Build and Aggergation):**
"Write Python code using pandas to clean the restaurants table casing, compute net revenue per order from payments (charges minus refunds), calculate delivery time in minutes from order_status_events, and aggregate into a gold weekly table grouped by week, cuisine, and city."

**What was accepted:** The logic for mapping payment transaction types to multipliers (charge = 1, refund = -1) and extracting time deltas between placed and delivered events.

**What was corrected:** Enforced explicit deduplication on the payments table (payment_id) before performing any group-by aggregation.

**Why:** Raw payment logs contained 60 duplicate rows. Without prior deduplication, joining the child payments table back to orders would cause duplicate transaction records to inflate net revenue calculations erroneously.

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

**Refund Attribution Timing:** Refunds are recorded in payments when they happen rather than retroactively shifting historical weekly revenue totals.

**External Disruptions:** Urban traffic conditions or severe weather across the five UK cities were not controlled for, meaning high delivery times are captured purely as raw timestamps without operational context.Add text here.

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.